# Quantization, LoRA, and Fitting in Memory

Every lesson so far has assumed the model fits. This one is about what to do when
it does not — and the techniques here are the reason you can run and adapt serious
models on a laptop at all.

Two ideas, both implemented from scratch:

**Quantization** stores weights in 8 or 4 bits instead of 32. A 4× to 8× reduction
in memory, with a quality cost you will measure rather than take on faith.

**LoRA** (Low-Rank Adaptation) freezes the pretrained weights and trains a pair of
tiny low-rank matrices alongside them. You will fine-tune a model to a completely
different author while training **under 2% of its parameters**.

No `bitsandbytes`, no `peft`, no Hugging Face — everything is plain PyTorch,
because building it is the point.

Read [Chapter 11 of the book](../book/index.html#ch11) alongside this.

**Runs on**: an M4 MacBook Air with 24 GB. Every number in the memory analysis is
calibrated to that machine.

## Step 1: Imports

In [ ]:
import copy
import math
import time

import nltk
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

device = torch.device('mps' if torch.backends.mps.is_available()
                      else 'cuda' if torch.cuda.is_available()
                      else 'cpu')
torch.manual_seed(1337)

print('torch  :', torch.__version__)
print('device :', device)

---
## Key Concept: The 16-Bytes-Per-Parameter Rule

Before any technique, the arithmetic that governs everything. Training one
parameter with Adam in float32 costs:

| Tensor | Bytes |
|--------|-------|
| weight | 4 |
| gradient | 4 |
| Adam `m` | 4 |
| Adam `v` | 4 |
| **total** | **16** |

Plus activations, which scale with batch size and sequence length.

So a 1-billion-parameter model needs **16 GB** just for optimizer state — before a
single activation. On 24 GB of unified memory shared with the OS, full fine-tuning
tops out around 1B parameters, and that is with nothing else running.

Inference is far cheaper: 4 bytes per parameter in fp32, 2 in fp16, 0.5 at 4-bit.
The gap between those two columns is what this lesson exploits.

In [ ]:
def memory_table(n_params):
    rows = [
        ('train, fp32 + Adam',     16),
        ('train, LoRA r=8 on fp16 base', 2.05),
        ('inference fp32',          4),
        ('inference fp16',          2),
        ('inference int8',          1),
        ('inference int4',          0.5),
    ]
    print(f'model: {n_params/1e6:,.0f}M parameters\n')
    print(f'{"mode":<32} {"bytes/param":>12} {"total":>10}')
    print('-' * 56)
    for name, b in rows:
        gb = n_params * b / 1e9
        fits = 'ok' if gb < 17 else 'TOO BIG'
        print(f'{name:<32} {b:>12.2f} {gb:>8.2f} GB  {fits}')


for n in (1e8, 1e9, 7e9):
    memory_table(n)
    print()

A 7-billion-parameter model cannot be fine-tuned on this machine by any
conventional means — 112 GB of optimizer state. At 4-bit inference it needs 3.5 GB
and runs comfortably. LoRA on a quantized base is what closes the gap between those
two facts, and that combination is what people mean by **QLoRA**.

---
## Step 2: A Model to Work On

We need something trained before compression means anything. This is the GPT from
[lesson 10](../10_gpt_from_scratch/solution.ipynb), condensed — if any of it is
unfamiliar, go back and build it there first.

In [ ]:
nltk.download('gutenberg', quiet=True)
from nltk.corpus import gutenberg

alice = gutenberg.raw('carroll-alice.txt')

chars = sorted(set(alice))
vocab_size = len(chars)
stoi = {c: i for i, c in enumerate(chars)}
itos = {i: c for c, i in stoi.items()}

def encode(s):
    # characters outside the Alice alphabet are dropped, which matters when we
    # bring in Shakespeare later
    return [stoi[c] for c in s if c in stoi]

def decode(ids):
    return ''.join(itos[i] for i in ids)

data = torch.tensor(encode(alice), dtype=torch.long)
n = int(0.9 * len(data))
train_data, val_data = data[:n], data[n:]

BLOCK_SIZE, BATCH_SIZE = 128, 64

def get_batch(split, source=None):
    src = source if source is not None else (train_data if split == 'train' else val_data)
    ix = torch.randint(len(src) - BLOCK_SIZE - 1, (BATCH_SIZE,))
    x = torch.stack([src[i:i + BLOCK_SIZE] for i in ix])
    y = torch.stack([src[i + 1:i + BLOCK_SIZE + 1] for i in ix])
    return x.to(device), y.to(device)

print(f'{len(alice):,} characters, {vocab_size} symbols')

In [ ]:
class CausalSelfAttention(nn.Module):
    def __init__(self, n_embd, n_head, block_size, dropout=0.1):
        super().__init__()
        self.n_head, self.d_k = n_head, n_embd // n_head
        self.qkv = nn.Linear(n_embd, 3 * n_embd, bias=False)
        self.proj = nn.Linear(n_embd, n_embd, bias=False)
        self.drop = nn.Dropout(dropout)
        self.register_buffer('mask', torch.triu(
            torch.ones(block_size, block_size, dtype=torch.bool), diagonal=1))

    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(C, dim=2)
        q = q.view(B, T, self.n_head, self.d_k).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.d_k).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.d_k).transpose(1, 2)
        att = (q @ k.transpose(-2, -1)) / math.sqrt(self.d_k)
        att = F.softmax(att.masked_fill(self.mask[:T, :T], float('-inf')), dim=-1)
        out = (att @ v).transpose(1, 2).contiguous().view(B, T, C)
        return self.drop(self.proj(out))


class Block(nn.Module):
    def __init__(self, n_embd, n_head, block_size, dropout=0.1):
        super().__init__()
        self.ln1 = nn.LayerNorm(n_embd)
        self.attn = CausalSelfAttention(n_embd, n_head, block_size, dropout)
        self.ln2 = nn.LayerNorm(n_embd)
        self.mlp = nn.Sequential(nn.Linear(n_embd, 4 * n_embd), nn.GELU(),
                                 nn.Linear(4 * n_embd, n_embd), nn.Dropout(dropout))

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        return x + self.mlp(self.ln2(x))


class GPT(nn.Module):
    def __init__(self, vocab_size, n_embd=128, n_head=4, n_layer=4,
                 block_size=BLOCK_SIZE, dropout=0.1):
        super().__init__()
        self.block_size = block_size
        self.tok_emb = nn.Embedding(vocab_size, n_embd)
        self.pos_emb = nn.Embedding(block_size, n_embd)
        self.blocks = nn.ModuleList(
            [Block(n_embd, n_head, block_size, dropout) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size, bias=False)
        self.lm_head.weight = self.tok_emb.weight
        self.apply(self._init)

    def _init(self, m):
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, std=0.02)
            if isinstance(m, nn.Linear) and m.bias is not None:
                nn.init.zeros_(m.bias)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        x = self.tok_emb(idx) + self.pos_emb(torch.arange(T, device=idx.device))
        for b in self.blocks:
            x = b(x)
        logits = self.lm_head(self.ln_f(x))
        loss = None if targets is None else F.cross_entropy(
            logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss


model = GPT(vocab_size).to(device)
n_params = sum(p.numel() for p in set(model.parameters()))
print(f'parameters: {n_params:,}')

In [ ]:
@torch.no_grad()
def evaluate(m, source=None, iters=40):
    m.eval()
    losses = []
    for _ in range(iters):
        x, y = get_batch('val', source)
        _, l = m(x, y)
        losses.append(l.item())
    m.train()
    return float(np.mean(losses))


def train(m, iters, lr=3e-4, source=None, label='train', log_every=300):
    opt = torch.optim.AdamW(
        [p for p in m.parameters() if p.requires_grad], lr=lr, weight_decay=0.1)
    warm = max(1, iters // 20)
    t0 = time.time()
    for step in range(iters):
        for g in opt.param_groups:
            if step < warm:
                g['lr'] = lr * (step + 1) / warm
            else:
                t = (step - warm) / max(1, iters - warm)
                g['lr'] = 0.1 * lr + 0.9 * lr * 0.5 * (1 + math.cos(math.pi * t))
        x, y = get_batch('train', source)
        _, loss = m(x, y)
        opt.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            [p for p in m.parameters() if p.requires_grad], 1.0)
        opt.step()
        if step % log_every == 0 or step == iters - 1:
            print(f'  [{label}] step {step:5d} | loss {loss.item():.4f}')
    print(f'  [{label}] done in {time.time() - t0:.1f}s')
    return m


print('Training the base model on Alice...')
train(model, 2000, label='base')
base_loss = evaluate(model)
print()
print(f'baseline val loss: {base_loss:.4f}   (perplexity {math.exp(base_loss):.2f})')

---
## Key Concept: What Quantization Actually Is

Take a tensor of floats and represent it with a small set of evenly spaced integer
levels. **Symmetric** quantization, the simplest scheme:

$$s = \frac{\max|W|}{2^{b-1}-1} \qquad q = \operatorname{round}(W/s) \qquad \hat{W} = q \cdot s$$

At 8 bits, `2^7 - 1 = 127` levels each side of zero. Store `q` as `int8` plus one
float scale, and you have replaced 4 bytes per weight with 1.

The error $|W - \hat{W}|$ is at most $s/2$, so **everything depends on `s`**, and
`s` is set by the single largest magnitude in the tensor. One outlier weight
stretches the grid for every other weight in the tensor. That single sentence
explains almost every technique in modern quantization research.

## Step 3: Quantize by Hand, First

**TODO:** implement symmetric per-tensor quantization from the formula above.

In [ ]:
def quantize_symmetric(w, bits=8):
    """Symmetric per-tensor quantization.

    Returns (q, scale) where q holds integers in [-(2^{b-1}-1), 2^{b-1}-1].
    """
    qmax = 2 ** (bits - 1) - 1
    # TODO: scale so that the largest magnitude maps to qmax
    scale = ...
    if scale == 0:
        scale = torch.tensor(1.0, device=w.device)
    # TODO: divide by the scale, round, and clamp to [-qmax, qmax]
    q = ...
    return q, scale


def dequantize(q, scale):
    # TODO: undo the scaling
    return ...


# A tiny worked example you can check against the book's by-hand exercise.
w = torch.tensor([0.0, 0.15, -0.42, 0.98, -0.07, 0.31])
q, s = quantize_symmetric(w, bits=8)
w_hat = dequantize(q, s)

print(f'scale s = max|w| / 127 = {w.abs().max():.4f} / 127 = {s:.6f}\n')
print(f'{"original":>10} {"q (int8)":>10} {"dequantized":>13} {"error":>10}')
for a, b_, c in zip(w, q, w_hat):
    print(f'{a:10.4f} {int(b_):10d} {c:13.6f} {abs(a - c):10.6f}')
print()
print(f'max error : {(w - w_hat).abs().max():.6f}   (bounded by s/2 = {s/2:.6f})')
print(f'RMSE      : {torch.sqrt(((w - w_hat)**2).mean()):.6f}')

---
## Step 4: The Outlier Problem, Measured

Now add a single large weight to the same tensor and watch what happens to the
other five. This is the entire motivation for per-channel scales.

In [ ]:
w_clean = torch.tensor([0.0, 0.15, -0.42, 0.98, -0.07, 0.31])
w_outlier = torch.cat([w_clean, torch.tensor([12.0])])   # one weight 12x larger

for name, wv in (('no outlier', w_clean), ('with outlier', w_outlier)):
    q, s = quantize_symmetric(wv, 8)
    err = (wv[:6] - dequantize(q, s)[:6]).abs()      # error on the ORIGINAL six only
    print(f'{name:>13}: scale={s:.6f}   RMSE on the first six weights = {err.pow(2).mean().sqrt():.6f}')

print()
print('The six ordinary weights did not change. Their quantization error grew by')
print('more than 12x because one unrelated weight widened the grid spacing.')
print()

# Per-channel scaling: one scale per output row, so an outlier only affects its own row.
def quantize_per_channel(w, bits=8):
    qmax = 2 ** (bits - 1) - 1
    scale = w.abs().amax(dim=1, keepdim=True) / qmax     # (out_features, 1)
    scale = torch.where(scale == 0, torch.ones_like(scale), scale)
    q = torch.clamp(torch.round(w / scale), -qmax, qmax)
    return q, scale


W = torch.randn(64, 128) * 0.1
W[7, 33] = 5.0                                            # one outlier in row 7

q_t, s_t = quantize_symmetric(W, 8)
q_c, s_c = quantize_per_channel(W, 8)

err_t = (W - dequantize(q_t, s_t)).pow(2).mean().sqrt()
err_c = (W - dequantize(q_c, s_c)).pow(2).mean().sqrt()

print(f'per-tensor  RMSE : {err_t:.6f}   (1 scale,  4 bytes of metadata)')
print(f'per-channel RMSE : {err_c:.6f}   ({W.shape[0]} scales, {W.shape[0]*4} bytes)')
print(f'improvement      : {err_t/err_c:.1f}x for {W.shape[0]*4} extra bytes on a '
      f'{W.numel()} weight tensor ({100*W.shape[0]*4/W.numel():.1f}% overhead)')
print()
print('This is why per-channel quantization is the default everywhere. The')
print('production version of the same insight — keeping a handful of outlier')
print('channels in fp16 while quantizing the rest — is LLM.int8().')

---
## Step 5: Quantize the Whole Model and Measure the Damage

Now the real experiment. Quantize every `nn.Linear` weight, dequantize back to
float (**fake quantization** — the standard way to measure quality without needing
integer kernels), and evaluate.

Embeddings and LayerNorm are left alone. LayerNorm has few parameters and high
sensitivity; embeddings are quantizable but are excluded here to isolate the effect
on the transformer weights.

In [ ]:
def quantize_model(m, bits=8, per_channel=True, skip_embeddings=True):
    """Return a copy with every Linear weight fake-quantized to `bits`."""
    mq = copy.deepcopy(m)
    n_quantized = n_weights = 0
    for module in mq.modules():
        if isinstance(module, nn.Linear):
            if skip_embeddings and module.weight.shape[0] == vocab_size:
                continue                       # tied lm_head shares the embedding
            w = module.weight.data
            if per_channel:
                q, s = quantize_per_channel(w, bits)
            else:
                q, s = quantize_symmetric(w, bits)
            module.weight.data = dequantize(q, s)
            n_quantized += 1
            n_weights += w.numel()
    return mq, n_quantized, n_weights


results = []
print(f'{"scheme":<26} {"val loss":>10} {"ppl":>8} {"Δ loss":>8} {"weight MB":>10}')
print('-' * 66)
print(f'{"fp32 baseline":<26} {base_loss:>10.4f} {math.exp(base_loss):>8.2f} '
      f'{0.0:>8.4f} {n_params*4/1e6:>10.2f}')

for bits, per_ch in [(8, True), (8, False), (6, True), (4, True), (4, False), (3, True), (2, True)]:
    mq, nq, nw = quantize_model(model, bits=bits, per_channel=per_ch)
    l = evaluate(mq)
    mb = (nw * bits / 8 + (n_params - nw) * 4) / 1e6
    label = f'{bits}-bit {"per-channel" if per_ch else "per-tensor"}'
    results.append((label, bits, per_ch, l, mb))
    print(f'{label:<26} {l:>10.4f} {math.exp(l):>8.2f} {l-base_loss:>8.4f} {mb:>10.2f}')

print()
print('Read the Δ loss column. 8-bit is essentially free. 4-bit per-channel costs')
print('something real but usable. Below 4 bits, per-tensor quantization falls apart')
print('entirely — and per-channel keeps working several bits longer.')

In [ ]:
labels = [r[0] for r in results]
losses = [r[3] for r in results]
sizes = [r[4] for r in results]

fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))

colors = ['#6c5ce7' if r[2] else '#e17055' for r in results]
ax[0].barh(range(len(labels)), [l - base_loss for l in losses], color=colors)
ax[0].set_yticks(range(len(labels))); ax[0].set_yticklabels(labels, fontsize=8)
ax[0].set_xlabel('increase in val loss (nats)')
ax[0].set_title('Quality cost of quantization')
ax[0].invert_yaxis(); ax[0].grid(alpha=0.3, axis='x')

ax[1].scatter(sizes, losses, c=colors, s=70, zorder=3)
ax[1].axhline(base_loss, color='#00b894', ls='--', lw=1.5, label='fp32 baseline')
for lbl, s, l in zip(labels, sizes, losses):
    ax[1].annotate(lbl.split()[0], (s, l), fontsize=7,
                   textcoords='offset points', xytext=(4, 4))
ax[1].set_xlabel('weight storage (MB)'); ax[1].set_ylabel('val loss')
ax[1].set_title('The actual trade-off')
ax[1].legend(fontsize=8); ax[1].grid(alpha=0.3)

plt.tight_layout(); plt.show()

---
### Concept Check: Quantization

1. Why does one outlier weight degrade the accuracy of every *other* weight in the
   same tensor?
2. Per-channel quantization stores one scale per output row instead of one per
   tensor. Why is that overhead negligible?
3. We fake-quantized: weights were converted to int and back to float. What does
   that measure correctly, and what does it fail to measure?

In [ ]:
# 1.
# 2.
# 3.

---
## Key Concept: LoRA

Fine-tuning updates every weight: $W' = W + \Delta W$, and $\Delta W$ is as large
as $W$. The observation behind LoRA is that $\Delta W$ — the *change* needed to
adapt a pretrained model to a new task — has very low intrinsic rank. It does not
need to be a full-rank matrix.

So parameterise it as a product of two thin matrices:

$$W' = W + \frac{\alpha}{r} BA \qquad B \in \mathbb{R}^{d \times r},\; A \in \mathbb{R}^{r \times d}$$

with $r \ll d$. Freeze $W$ entirely and train only $A$ and $B$. Instead of $d^2$
parameters you train $2dr$ — at $d=128$, $r=8$ that is 2048 instead of 16384, a
factor of 8; at $d=4096$, $r=8$ it is 65k instead of 16.7M, a factor of 256.

Three details make it work:

- **$B$ is initialised to zero.** So $BA = 0$ at step 0 and the adapted model is
  *exactly* the base model. Training starts from the pretrained behaviour rather
  than from a perturbation of it.
- **$A$ is initialised randomly.** Both zero would leave the gradient zero forever.
- **$\alpha/r$ scaling** keeps the update magnitude roughly constant as you change
  the rank, so retuning $r$ does not force retuning the learning rate.

The optimizer state shrinks by the same factor as the parameters, which is what
actually makes this fit in memory.

## Step 6: Implement LoRA

**TODO:** complete the `LoRALinear` wrapper.

In [ ]:
class LoRALinear(nn.Module):
    """Wraps a frozen nn.Linear with a trainable low-rank update."""

    def __init__(self, base: nn.Linear, r=8, alpha=16, dropout=0.0):
        super().__init__()
        self.base = base
        # TODO: freeze every parameter of the base layer
        for p in self.base.parameters():
            ...

        self.r = r
        self.scaling = alpha / r
        in_f, out_f = base.in_features, base.out_features

        # TODO: create A of shape (r, in_f) and B of shape (out_f, r), both as
        # nn.Parameter. B must start at exactly zero; A must not.
        # Build them on base.weight.device — the base model is already on the GPU,
        # and parameters created on the CPU default will fail at the first matmul.
        dev = base.weight.device
        self.A = ...
        self.B = ...
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        # TODO: base output + scaling * (x @ A.T @ B.T)
        return ...

    def merged_weight(self):
        """W + (alpha/r) B A — for folding the adapter back into the base."""
        return self.base.weight.data + (self.B @ self.A) * self.scaling


# Verify the defining property: at initialisation the wrapper is a no-op.
lin = nn.Linear(16, 16, bias=False)
lora = LoRALinear(copy.deepcopy(lin), r=4)
x = torch.randn(2, 16)
print('identical to the base layer at step 0:',
      torch.allclose(lin(x), lora(x), atol=1e-6))
print(f'B is all zeros: {bool((lora.B == 0).all())}   A is not: {bool((lora.A != 0).any())}')

---
## Step 7: Adapt Alice to Shakespeare

The real test. Take the model trained on Carroll's prose and adapt it to
*Hamlet* — different vocabulary, different rhythm, verse instead of prose —
while training under 2% of its parameters.

In [ ]:
hamlet = gutenberg.raw('shakespeare-hamlet.txt')
shake_data = torch.tensor(encode(hamlet), dtype=torch.long)
n_s = int(0.9 * len(shake_data))
shake_train, shake_val = shake_data[:n_s], shake_data[n_s:]

print(f'Hamlet: {len(hamlet):,} characters, {len(shake_data):,} usable after '
      f'dropping symbols outside the Alice alphabet')
print()
print(repr(hamlet[1000:1200]))
print()

# How badly does the Alice model do on Shakespeare, before adaptation?
shake_before = evaluate(model, shake_val)
print(f'Alice model on Alice     : {base_loss:.4f}  (ppl {math.exp(base_loss):.2f})')
print(f'Alice model on Shakespeare: {shake_before:.4f}  (ppl {math.exp(shake_before):.2f})')
print(f'domain gap               : {shake_before - base_loss:+.4f} nats')

In [ ]:
def apply_lora(m, r=8, alpha=16, targets=('qkv', 'proj')):
    """Replace the named Linear submodules of every block with LoRALinear."""
    for p in m.parameters():
        p.requires_grad = False                  # freeze everything first
    n = 0
    for block in m.blocks:
        for name in targets:
            if hasattr(block.attn, name):
                setattr(block.attn, name, LoRALinear(getattr(block.attn, name), r, alpha))
                n += 1
    return m.to(device), n


lora_model = copy.deepcopy(model).to(device)
lora_model, n_adapted = apply_lora(lora_model, r=8, alpha=16)

trainable = sum(p.numel() for p in lora_model.parameters() if p.requires_grad)
total = sum(p.numel() for p in set(lora_model.parameters()))

print(f'adapted {n_adapted} linear layers')
print(f'trainable : {trainable:,}')
print(f'total     : {total:,}')
print(f'fraction  : {100 * trainable / total:.2f}%')
print()
print(f'optimizer state: {trainable * 8 / 1024:.1f} KB with LoRA vs '
      f'{total * 8 / 1024:.1f} KB for a full fine-tune')
print()
print('sanity: the adapted model must score exactly the base model on Alice')
print(f'  base       : {base_loss:.4f}')
print(f'  lora (t=0) : {evaluate(lora_model):.4f}   <- B=0 means no change yet')

In [ ]:
print('LoRA fine-tuning on Hamlet (only A and B receive gradients)...')
train(lora_model, 800, lr=1e-3, source=shake_train, label='lora', log_every=200)

lora_shake = evaluate(lora_model, shake_val)
lora_alice = evaluate(lora_model)

print()
print(f'{"":<34}{"on Shakespeare":>16}{"on Alice":>12}')
print('-' * 62)
print(f'{"base model (Alice only)":<34}{shake_before:>16.4f}{base_loss:>12.4f}')
print(f'{"+ LoRA r=8, 800 steps":<34}{lora_shake:>16.4f}{lora_alice:>12.4f}')
print(f'{"improvement on target domain":<34}{shake_before - lora_shake:>+16.4f}')
print()
print(f'perplexity on Shakespeare: {math.exp(shake_before):.2f} -> {math.exp(lora_shake):.2f}')
print()
print(f'Achieved by training {trainable:,} parameters — {100*trainable/total:.2f}% of the model.')
print('Note the regression on Alice: adapting to a new domain costs performance on')
print('the old one. That is catastrophic forgetting, and it is why LoRA adapters are')
print('usually kept as separate swappable files rather than merged permanently.')

---
## Step 8: Rank Sweep

How low can `r` go? This is the question LoRA's whole premise rests on.

In [ ]:
rank_results = []
for r in (1, 2, 4, 8, 16, 32):
    m_r = copy.deepcopy(model).to(device)
    m_r, _ = apply_lora(m_r, r=r, alpha=2 * r)
    tp = sum(p.numel() for p in m_r.parameters() if p.requires_grad)
    train(m_r, 500, lr=1e-3, source=shake_train, label=f'r={r}', log_every=10**9)
    l = evaluate(m_r, shake_val)
    rank_results.append((r, tp, l))
    print(f'  r={r:<3} trainable={tp:>7,} ({100*tp/total:5.2f}%)  '
          f'val loss={l:.4f}  ppl={math.exp(l):.2f}')

print()
print(f'no adaptation at all: {shake_before:.4f}')
print()

# Report the gain as a FRACTION of what the largest rank achieved, which is the
# number the low-rank hypothesis is actually about.
best_gain = shake_before - min(l for _, _, l in rank_results)
print(f'{"rank":>5} {"trainable":>10} {"% params":>9} {"val loss":>9} {"gain":>7} {"% of max gain":>14}')
for r, tp, l in rank_results:
    gain = shake_before - l
    print(f'{r:>5} {tp:>10,} {100*tp/total:>8.2f}% {l:>9.4f} {gain:>7.3f} {100*gain/best_gain:>13.0f}%')

In [ ]:
rs = [r[0] for r in rank_results]
tps = [r[1] for r in rank_results]
ls = [r[2] for r in rank_results]

fig, ax = plt.subplots(1, 2, figsize=(11.5, 3.6))

ax[0].plot(rs, ls, 'o-', color='#6c5ce7', lw=2)
ax[0].axhline(shake_before, color='#d63031', ls='--', lw=1.5, label='no adaptation')
ax[0].set_xscale('log', base=2); ax[0].set_xticks(rs); ax[0].set_xticklabels(rs)
ax[0].set_xlabel('LoRA rank r'); ax[0].set_ylabel('val loss on Shakespeare')
ax[0].set_title('Quality vs rank')
ax[0].legend(fontsize=8); ax[0].grid(alpha=0.3)

ax[1].plot([100*t/total for t in tps], ls, 'o-', color='#00b894', lw=2)
for r, t, l in rank_results:
    ax[1].annotate(f'r={r}', (100*t/total, l), fontsize=7,
                   textcoords='offset points', xytext=(4, 4))
ax[1].set_xlabel('trainable parameters (% of model)')
ax[1].set_ylabel('val loss on Shakespeare')
ax[1].set_title('Quality vs cost')
ax[1].grid(alpha=0.3)

plt.tight_layout(); plt.show()

print('Read the last column honestly. Rank 1 — a few thousand parameters — already')
print('captures well over half the achievable gain, which is strong evidence for the')
print('low-rank hypothesis: if adaptation needed a full-rank update, one direction')
print('would achieve almost nothing.')
print()
print('But the curve has NOT plateaued: the largest rank is still measurably better')
print('than r=8. The honest description is steep diminishing returns, not saturation.')
print('Where you stop is a budget decision, not a point where rank stops helping.')
print()
print('Caveat: a 0.8M-parameter model, 500 steps, and a large domain shift all push')
print('towards needing MORE rank than a realistic setting. On large models, where the')
print('base already contains most of what the target task needs, the curve flattens')
print('considerably earlier.')

---
## Step 9: QLoRA — Both at Once

The combination that makes large-model fine-tuning possible on consumer hardware:

1. Quantize the frozen base to 4 bits — it is never updated, so quantization error
   is a fixed, tolerable bias rather than something that compounds through training.
2. Keep the LoRA adapters in full precision — they are tiny, and they are what
   actually learns.
3. Gradients flow *through* the quantized weights to the adapters. Nothing needs
   to backpropagate *into* the quantized values.

This is what lets a 65B model be fine-tuned on a single 48 GB GPU, and it is what
would let you adapt a 7B model on this laptop.

In [ ]:
# 4-bit base + fp32 adapters
qlora = copy.deepcopy(model).to(device)
qlora, _, _ = quantize_model(qlora, bits=4, per_channel=True)   # returns (model, n_layers, n_weights)
qlora, _ = apply_lora(qlora, r=8, alpha=16)

print(f'base on Shakespeare, 4-bit + untrained adapters: {evaluate(qlora, shake_val):.4f}')
print()
print('QLoRA fine-tuning...')
train(qlora, 800, lr=1e-3, source=shake_train, label='qlora', log_every=400)

qlora_loss = evaluate(qlora, shake_val)

print()
print(f'{"configuration":<38}{"Shakespeare loss":>18}{"weight MB":>11}')
print('-' * 68)
print(f'{"base fp32, no adaptation":<38}{shake_before:>18.4f}{n_params*4/1e6:>11.2f}')
print(f'{"fp32 base + LoRA r=8":<38}{lora_shake:>18.4f}{n_params*4/1e6:>11.2f}')
print(f'{"4-bit base + LoRA r=8 (QLoRA)":<38}{qlora_loss:>18.4f}'
      f'{n_params*0.5/1e6 + trainable*4/1e6:>11.2f}')
print()
print('QLoRA gives up a little quality against fp32 LoRA and stores the frozen base')
print('at an eighth of the size. At this scale the saving is a rounding error; at 7B')
print('parameters it is the difference between 28 GB and 3.5 GB, which is the')
print('difference between impossible and routine on a 24 GB machine.')

In [ ]:
@torch.no_grad()
def sample(m, prompt, n=220, temperature=0.8):
    m.eval()
    idx = torch.tensor([encode(prompt)], dtype=torch.long, device=device)
    for _ in range(n):
        logits, _ = m(idx[:, -m.block_size:])
        probs = F.softmax(logits[:, -1, :] / temperature, dim=-1)
        idx = torch.cat([idx, torch.multinomial(probs, 1)], dim=1)
    m.train()
    return decode(idx[0].tolist())


for name, m in (('base model (Alice)', model),
                ('LoRA-adapted (Hamlet)', lora_model),
                ('QLoRA 4-bit (Hamlet)', qlora)):
    print('=' * 76)
    print(name)
    print('=' * 76)
    print(sample(m, 'Ham. '))
    print()

---
### Concept Check: LoRA and QLoRA

1. Why is `B` initialised to zero rather than randomly?
2. LoRA trains ~2% of the parameters. Does it also use ~2% of the memory during
   training? Explain what does and does not shrink.
3. In QLoRA the base weights are quantized to 4 bits and frozen. Why is
   quantization error more tolerable there than it would be in a model you were
   fully fine-tuning?

In [ ]:
# 1.
# 2.
# 3.

---
## Step 11: The Same Two Ideas, With the Libraries

You implemented quantization and LoRA from their definitions. Both exist as one-liners,
and the point of having built them is that you can now check the libraries rather than
trust them — and debug them when they misbehave.

### Quantization: `torch.ao.quantization`

Ours was **fake** quantization: values went to integers and back to float, which
measures the accuracy cost exactly but delivers no real saving. PyTorch's dynamic
quantization stores genuinely int8 weights and computes in int8.

Three catches, all of which cost real time to discover:

1. **The int8 kernels are CPU-only.** The model has to come off MPS first.
2. **The quantization engine is not set by default on this build.** Calling
   `quantize_dynamic` straight away fails with
   `RuntimeError: Didn't find engine for operation quantized::linear_prepack NoQEngine`,
   which does not obviously mean "pick a backend". On Apple silicon the available
   engine is `qnnpack` and you must select it yourself.
3. **`torch.ao.quantization` is deprecated** in current PyTorch in favour of the
   separate `torchao` package (`quantize_` for eager mode, `prepare_pt2e` /
   `convert_pt2e` for graph mode). It still works, and it is still the clearest
   illustration of the idea, but check the migration notes before building on it.

The broader lesson is the one from the KV cache in lesson 10: **a technique working
is not the same as a technique working on your hardware.** Quantized inference paths
are unusually hardware-specific, and on this machine the stacks that actually deliver
end-to-end speedups are MLX and llama.cpp/GGUF.

In [ ]:
import io
import warnings

with warnings.catch_warnings():
    warnings.simplefilter('ignore')          # the module emits a deprecation notice
    import torch.ao.quantization as tq

# Pick a backend explicitly. Without this the engine is 'none' and the first
# quantized op raises NoQEngine.
print('supported engines :', torch.backends.quantized.supported_engines)
print('default engine    :', torch.backends.quantized.engine)

engine = next((e for e in ('qnnpack', 'fbgemm', 'x86')
               if e in torch.backends.quantized.supported_engines), None)
if engine:
    torch.backends.quantized.engine = engine
    print('selected engine   :', torch.backends.quantized.engine)
else:
    print('no quantization engine available on this build')
print()


def state_dict_bytes(m):
    """Actual serialised size — the number that matters for shipping a model."""
    buf = io.BytesIO()
    torch.save(m.state_dict(), buf)
    return buf.getbuffer().nbytes


cpu_model = copy.deepcopy(model).cpu().eval()

# real dynamic int8 quantization of every Linear
qmodel = None
if engine:
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        qmodel = tq.quantize_dynamic(cpu_model, {nn.Linear}, dtype=torch.qint8)

if qmodel is None:
    print('Skipping: no quantization engine. The fake-quantization results above')
    print('still hold — they measure accuracy, which is hardware-independent.')
    raise SystemExit if False else None

fp32_bytes = state_dict_bytes(cpu_model)
int8_bytes = state_dict_bytes(qmodel) if qmodel is not None else fp32_bytes

print(f'fp32 state_dict : {fp32_bytes/1e6:7.3f} MB')
print(f'int8 state_dict : {int8_bytes/1e6:7.3f} MB')
print(f'ratio           : {fp32_bytes/int8_bytes:7.2f}x smaller  (REAL bytes, not simulated)')
print()

# What did each Linear become? Print the fully qualified type: the quantized
# class is ALSO called 'Linear', so the bare name makes it look like nothing
# happened. The module path is what tells you.
def linear_kinds(m):
    kinds = {}
    for _, mod in m.named_modules():
        if 'Linear' in type(mod).__name__:
            key = type(mod).__module__ + '.' + type(mod).__name__
            kinds[key] = kinds.get(key, 0) + 1
    return kinds

print('before:')
for k, v in linear_kinds(cpu_model).items():
    print(f'  {v:>3} x {k}')
print('after:')
for k, v in linear_kinds(qmodel or cpu_model).items():
    print(f'  {v:>3} x {k}')
print()
print('Note the embedding is untouched — quantize_dynamic only converts the module')
print('types you list, which is the same choice we made by hand in quantize_model().')

In [ ]:
# Compare quality: ours (fake, per-channel) vs PyTorch's (real, per-tensor dynamic).
@torch.no_grad()
def evaluate_cpu(m, iters=20):
    m.eval()
    losses = []
    for _ in range(iters):
        x, y = get_batch('val')
        _, l = m(x.cpu(), y.cpu())
        losses.append(l.item())
    return float(np.mean(losses))


ours8, _, _ = quantize_model(model, bits=8, per_channel=True)

print(f'{"model":<38}{"val loss":>10}{"Δ":>9}')
print('-' * 58)
base_cpu = evaluate_cpu(cpu_model)
print(f'{"fp32 baseline (cpu)":<38}{base_cpu:>10.4f}{0.0:>9.4f}')

ours_loss = evaluate_cpu(copy.deepcopy(ours8).cpu())
print(f'{"ours: fake int8, per-channel":<38}{ours_loss:>10.4f}{ours_loss-base_cpu:>9.4f}')

if qmodel is not None:
    theirs_loss = evaluate_cpu(qmodel)
    print(f'{"torch.ao: real int8, dynamic":<38}{theirs_loss:>10.4f}{theirs_loss-base_cpu:>9.4f}')

print()
print('Close, and the small gap is informative rather than mysterious: our version')
print('uses per-CHANNEL scales while quantize_dynamic uses one scale per tensor for')
print('the weights. Exercise 00.5 and Step 4 showed exactly what that costs. You can')
print('read the difference because you built both.')

---
### LoRA: the `peft` library

`peft` is the standard, and its API maps one-to-one onto what you wrote:

```python
from peft import LoraConfig, get_peft_model

model = get_peft_model(base_model, LoraConfig(
    r=8,                                 # your r
    lora_alpha=16,                       # your alpha, same alpha/r scaling
    target_modules=["qkv", "proj"],      # your `targets` argument
    lora_dropout=0.05,
    bias="none",
))

model.print_trainable_parameters()       # your trainable/total line
model.save_pretrained("adapter/")        # a few hundred KB
model = model.merge_and_unload()         # your merged_weight(), applied everywhere
```

It is not installed here, because everything in this curriculum runs on `uv sync` with
no extra dependencies. But the one operation worth verifying is `merge_and_unload` —
folding `B@A` back into the frozen weight — because it is what you do before shipping,
and because getting the scaling wrong there is a real and silent bug.

In [ ]:
# Verify the merge: a merged layer must be numerically identical to the wrapped one.
lin = nn.Linear(32, 32, bias=False).to(device)
wrapped = LoRALinear(copy.deepcopy(lin), r=4, alpha=8).to(device)

# give the adapter some non-trivial content, as training would
with torch.no_grad():
    wrapped.B.normal_(0, 0.02)

x = torch.randn(5, 32, device=device)
out_adapter = wrapped(x)

merged = nn.Linear(32, 32, bias=False).to(device)
with torch.no_grad():
    merged.weight.copy_(wrapped.merged_weight())
out_merged = merged(x)

print(f'max difference : {(out_adapter - out_merged).abs().max().item():.3e}')
print(f'allclose       : {torch.allclose(out_adapter, out_merged, atol=1e-5)}')
print()
print('Merging costs nothing at inference and removes the adapter indirection — but')
print('it also destroys the thing that made adapters useful, since you can no longer')
print('swap tasks without reloading the base. Merge to ship one specialised model;')
print('keep them separate to serve many.')
print()

# the scaling bug this check catches
with torch.no_grad():
    wrong = wrapped.base.weight.data + (wrapped.B @ wrapped.A)   # alpha/r omitted
print(f'if you forget the alpha/r factor, the merged weight is off by {wrapped.scaling}x')
print(f'  correct merge norm : {wrapped.merged_weight().norm():.4f}')
print(f'  buggy merge norm   : {wrong.norm():.4f}')
print('Both run. Only one is your model.')

---
## Step 12: Other Techniques Worth Knowing

Things that matter in practice, in rough order of how often you will reach for them.

**Mixed precision (fp16 / bf16).** Store and compute in 16 bits, keep a float32
master copy of the weights. Roughly halves memory and speeds up matrix
multiplication. `bf16` has the same exponent range as fp32 and so needs no loss
scaling, which makes it the safer default where supported. On MPS, fp16 inference
is well supported; mixed-precision *training* is less mature than on CUDA.

**Gradient checkpointing.** Discard intermediate activations during the forward
pass and recompute them during the backward pass. Cuts activation memory by roughly
√N for N layers at about 30% extra compute. `torch.utils.checkpoint`.

**Gradient accumulation.** Simulate a large batch by running several small ones and
calling `step()` only at the end. Free, since gradients accumulate by default —
remember to scale the loss.

**Flash attention.** Tiles the attention computation so the L×L score matrix is
never materialised in memory. `F.scaled_dot_product_attention` uses it where
available, and is worth preferring over a hand-written implementation in production
code.

**Knowledge distillation.** Train a small model to match a large model's output
distribution. Complementary to everything above, and often better than quantizing
a large model very aggressively.

**Structured pruning.** Remove whole attention heads or FFN channels. Unlike
unstructured sparsity, it produces genuinely smaller dense matrices and so gives
real speedups without special kernels.

In [ ]:
n_layer, n_embd, n_head = 4, 128, 4
seq, batch = BLOCK_SIZE, BATCH_SIZE

act_per_layer = batch * seq * n_embd * 4          # bytes, roughly
attn_matrix = batch * n_head * seq * seq * 4

print('Activation memory for THIS model, per forward pass:')
print(f'  hidden states : {n_layer * act_per_layer / 1e6:8.2f} MB')
print(f'  attention     : {n_layer * attn_matrix / 1e6:8.2f} MB')
print()
print('Now scale the sequence length, holding everything else fixed:')
print(f'{"seq len":>9} {"hidden MB":>11} {"attention MB":>14} {"total MB":>10}')
for s in (128, 256, 512, 1024, 2048, 4096):
    h = n_layer * batch * s * n_embd * 4 / 1e6
    a = n_layer * batch * n_head * s * s * 4 / 1e6
    print(f'{s:>9} {h:>11.1f} {a:>14.1f} {h+a:>10.1f}')
print()
print('Hidden states grow linearly; attention grows quadratically and takes over')
print('completely. At 4096 tokens this 0.8M-parameter model would need more than')
print('20 GB of activations — the *model* is irrelevant to the memory problem.')
print('That is what flash attention and sliding-window attention exist to fix.')

---
## What You Built

- **Symmetric quantization** from the formula, and a measurement of the outlier
  problem that motivates per-channel scales
- A **bit-width sweep** across the whole model showing where quality actually breaks
- **LoRA** from scratch, with the zero-init property verified
- A real **domain adaptation** — Carroll to Shakespeare — training under 2% of the
  parameters
- A **rank sweep** testing LoRA's central low-rank claim empirically
- **QLoRA**: a 4-bit frozen base with full-precision adapters
- An **activation-memory analysis** showing why sequence length, not parameter
  count, is usually what runs you out of memory
- The **library equivalents** — `torch.ao.quantization` measured against your fake
  quantization on real serialised bytes, and the `peft` merge verified numerically

## Next

[Lesson 12](../12_capstone_projects/prompt.ipynb) has no TODOs to fill in. It sets
four open-ended projects — a translator, a text generator, a domain-adapted
assistant, and an architecture comparison — that combine everything from lessons 00
through 11.